# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamed-6513/flyrank_ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** One single web page (`content_id`).

**Time Window:** We will use a 30-day window. We will compare the most recent 30 days of data (our label window) against the 30 days prior to that (our feature window) to detect if traffic is decaying.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Feature:** `word_count`, `search_volume`, `clicks_prev_30d` (Data we know BEFORE we need to make the prediction).
- **Label:** `clicks_last_30d` (The actual outcome we are training the AI to predict).
- **Context:** `content_id`, `client_id` (IDs used only for grouping and organizing, NOT for learning).
- **Excluded:** `trend_direction`, `trend_pct`, `is_declining_label` (Must be entirely hidden from the AI because they contain the final answer. Including them would cause target leakage/cheating).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    hf_token = os.environ.get('HF_TOKEN')

if not hf_token:
    raise ValueError('HF_TOKEN not found. Set it in Colab Secrets or as an environment variable.')

import duckdb
import pandas as pd

# Connect to DuckDB and set up HTTP/HF extension
con = duckdb.connect()
con.execute('INSTALL httpfs;')
con.execute('LOAD httpfs;')
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# 1. Grain and Counts Check (dim_content)
print('--- Grain & Count Check: dim_content ---')
query_grain = """
SELECT COUNT(*) as total_rows 
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
"""
display(con.execute(query_grain).df())

# 2. Windows Check (Testing on the March 2026 partition to save bandwidth)
print('\n--- Time Windows Check (March 2026 partition) ---')
query_windows = """
SELECT MIN(report_date) as start_date, MAX(report_date) as end_date 
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
display(con.execute(query_windows).df())

# 3. Missingness check on a specific column (e.g. search_volume in dim_content)
print('\n--- Missingness Check (search_volume) ---')
query_missing = """
SELECT AVG(CASE WHEN search_volume IS NULL THEN 1.0 ELSE 0 END) * 100 as pct_missing_search_volume
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
"""
display(con.execute(query_missing).df())


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Data Limits to Remember:**
1. **Unbalanced History:** Some clients have 17 months of history, others have only a few. We must check `gsc_data_start` per client instead of assuming everyone starts on the same date.
2. **GA4 Missingness:** Rows before a client's `ga4_data_start` have zeros for GA4 columns, but this means 'no tracking', not 'zero engagement'. We must filter using `ga4_data_available = TRUE`.
3. **Missing Keyword Data by Type:** `search_volume` and `competition` are systematically missing for certain content types (like news feeds) because they don't target keywords.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.